# 02 · Evaluation and quality

How this project keeps an LLM honest, and how a regression gets caught **before** it ships.

Four layers, each answering a different question:

| Layer | Question | Command |
| --- | --- | --- |
| **Golden gate** | Does retrieval still surface the right evidence, and does it abstain when it should? | `make gate` (blocks CI) |
| **RAGAS** | Are the *answers* faithful and relevant, judged independently? | `make ragas` |
| **Ablation** | Does each retrieval stage (dense → hybrid → rerank) earn its cost? | `make ablation` |
| **Drift** | Is production traffic drifting from what we validated on? | `make drift` |

The gate and drift cells below run **offline on recorded fixtures and deterministic fakes** — no
keys, no network — exactly as they run in CI. (Run with the notebook deps: `uv run --extra notebook
jupyter lab`.)

## 1. The golden gate

Five fixtures over a small recorded corpus: four in-domain questions that must *retrieve* the right
fact, and one out-of-domain question that must *abstain*. `min_score = 1.0`, so **one** failure
blocks a merge.

In [ ]:
import os, tempfile, copy
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from evaluation.ci_gate import load_gate, run_gate

gate = load_gate("../evaluation/fixtures/gate.json")
tmp = tempfile.mkdtemp()
healthy = run_gate(gate, min_score=1.0, trace_path=os.path.join(tmp, "g1.jsonl"))
print("fixtures:", [f"{f['id']}:{f['expect']}" for f in gate["fixtures"]])
print(f"gate score = {healthy['score']}  ->  {'PASS (merge allowed)' if healthy['passed'] else 'BLOCKED'}")

## 2. Catching a regression

Simulate a real regression: retrieval breaks (the corpus goes missing), so in-domain questions
wrongly abstain — and the gate **blocks the merge** instead of silently shipping a mute assistant.

![the gate blocks a retrieval regression](../docs/img/eval-gate.png)

In [ ]:
broken = copy.deepcopy(gate)
broken["corpus"] = []  # retrieval finds nothing -> in-domain questions wrongly abstain
regressed = run_gate(broken, min_score=1.0, trace_path=os.path.join(tmp, "g2.jsonl"))
print(f"gate score = {regressed['score']}  ->  {'PASS' if regressed['passed'] else 'BLOCKED (merge stopped)'}")
for r in regressed["results"]:
    print(f"  {r['id']} expect={r['expect']:9s} passed={r['passed']}")

fig, ax = plt.subplots(figsize=(4.6, 3))
s = [healthy["score"], regressed["score"]]
b = ax.bar(["healthy", "regressed"], s, color=["#1f6f5c", "#d1495b"])
ax.axhline(1.0, ls="--", c="#888", lw=1); ax.set_ylim(0, 1.1); ax.set_ylabel("gate score")
ax.set_title("The eval gate blocks a retrieval regression")
for r, v in zip(b, s): ax.text(r.get_x()+r.get_width()/2, v+0.02, f"{v:.1f}", ha="center", weight="bold")
plt.tight_layout(); plt.show()

## 3. Drift monitoring

The gate protects the code; **drift** protects against the world changing under a frozen model.
The Population Stability Index (PSI) compares a recent window against the validated baseline — a
large PSI is an early warning to re-evaluate, long before answer quality visibly drops.

![drift monitor catches a distribution shift](../docs/img/drift-psi.png)

In [ ]:
import random
from mlops.drift import psi
rng = random.Random(7)
baseline = [rng.gauss(900, 120) for _ in range(400)]     # validated latency (ms)
stable   = [rng.gauss(905, 122) for _ in range(400)]     # same distribution
shifted  = [rng.gauss(1300, 180) for _ in range(400)]    # upstream slowdown
print(f"PSI (stable week)  = {psi(baseline, stable):.3f}  -> no drift")
print(f"PSI (latency shift)= {psi(baseline, shifted):.3f}  -> DRIFT flagged (> 0.2)")

## 4. RAGAS and ablation (the live half)

Two more layers run against the live stack (keys + an ingest):

- **`make ragas`** scores *answer* faithfulness, relevancy, and context precision/recall with an
  **independent judge model**, so a weak answer is not graded by the model that wrote it.
- **`make ablation`** writes [`docs/eval-report.md`](../docs/eval-report.md) comparing dense-only vs
  hybrid vs hybrid+rerank on `hit@k` / `mrr`, so the reranker has to *earn* its cost.
- **`make promote`** gates a config through MLflow dev → staging → prod on the gate score.

Together: the code can't regress silently (gate), the answers are judged independently (RAGAS),
every retrieval stage is justified (ablation), and production is watched (drift).